# Resume Dataset. What the job title is worth, and how search does on this data

I am building a pet project called [CV Screener](https://ananasdda.github.io/cv-screener/). It is a local search over résumés where exact filters on fields work together with semantic search. I built it on 14 synthetic résumés and wanted to see how the semantic part behaves on real ones. This dataset fits, it has 2,483 résumés in 24 categories. The project code is [on GitHub](https://github.com/ananasDDA/cv-screener), and this notebook uses the same embedding model and the same library as the project.

A fine-tuned BERT gets about 92% on this data, and you will not see that number here. I do not fine-tune anything. I wanted to know what a small embedding model can do out of the box, where it goes wrong and what helps it.

In [ ]:
!pip -q install fastembed

In [ ]:
import csv, re, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

SAMPLE = None   # set to e.g. 12 to keep 12 résumés per category for a quick dry run
SEED = 7

def table(rows, cols=None):
    cols = cols or list(rows[0])
    head = "".join(f"<th style='text-align:left;padding:3px 10px'>{c}</th>" for c in cols)
    body = "".join("<tr>" + "".join(f"<td style='padding:3px 10px'>{r.get(c, '')}</td>" for c in cols) + "</tr>" for r in rows)
    display(HTML(f"<table><tr>{head}</tr>{body}</table>"))

## 1. The data

In [ ]:
found = sorted(Path("/kaggle/input").rglob("Resume.csv")) + sorted(Path(".").rglob("Resume.csv"))
assert found, "Resume.csv not found. Attach the dataset with Add Input, search for snehaanbhawal/resume-dataset"
csv_path = found[0]
print("reading", csv_path)
csv.field_size_limit(sys.maxsize)
with csv_path.open(encoding="utf-8") as fh:
    rows = [r for r in csv.DictReader(fh) if r["Resume_str"].strip()]
if SAMPLE:
    seen, kept = {}, []
    for r in rows:
        if seen.get(r["Category"], 0) < SAMPLE:
            kept.append(r); seen[r["Category"]] = seen.get(r["Category"], 0) + 1
    rows = kept
raw = [r["Resume_str"] for r in rows]
y = [r["Category"] for r in rows]
cats = sorted(set(y))
sizes = {c: y.count(c) for c in cats}
print(len(raw), "résumés,", len(cats), "categories")
print("smallest", min(sizes, key=sizes.get), sizes[min(sizes, key=sizes.get)], "| largest", max(sizes, key=sizes.get), sizes[max(sizes, key=sizes.get)])
print("median length", int(np.median([len(t) for t in raw])), "characters | chance level", f"{1/len(cats):.0%}")

There are 24 categories. The smallest is BPO with 22 résumés and the largest is Information Technology with 120. A median résumé is 5,900 characters long. Guessing at random gives 5%.

Almost every résumé starts with a job title. In 65% of them the title contains a word from the category name, like HR Administrator in HR or Staff Consultant in Consultant. A category here is the section of the site where the person posted the résumé, so the label is more or less derived from the title. This matters later.

In [ ]:
def title_of(text):
    return re.split(r"\s{3,}", text.strip(), maxsplit=1)[0].lower()

def category_words(cat):
    return [w for w in re.split(r"[-\s]+", cat.lower()) if w]

hit = {c: 0 for c in cats}
for t, c in zip(raw, y):
    hit[c] += any(re.search(rf"\b{re.escape(w)}", title_of(t)) for w in category_words(c))
print(f"title contains a category word in {sum(hit.values()) / len(y):.0%} of résumés")
order = sorted(cats, key=lambda c: -hit[c] / sizes[c])
table([{"category": c, "résumés": sizes[c], "title gives it away": f"{hit[c] / sizes[c]:.0%}"} for c in order[:6] + order[-6:]])

## 2. What has been done on this dataset

Most notebooks on the dataset page predict the category. The usual recipe is to clean the text, turn it into a bag of words or a TF-IDF matrix with a few hundred features, and train a random forest, logistic regression or k-NN. Their authors report between 53% and 68%. The most upvoted classifier gets 84% on the training set and 53% on the test set.

Outside Kaggle there is a [detailed write-up](https://marcocamilo.com/portfolio/resume-classifier.html) where a linear SVM on TF-IDF reaches 87% and a fine-tuned BERT reaches 92%. Classes were balanced there by resampling, and I could not tell from the text whether that happened before or after the test split.

Nobody I read removes the job title from the text.

In [ ]:
others = [
    ("Kaggle, random forest on word counts", 0.53),
    ("Kaggle, TF-IDF(800) + k-NN", 0.558),
    ("Kaggle, TF-IDF(800) + logistic regression", 0.634),
    ("Kaggle, TF-IDF(800) + random forest", 0.681),
    ("Blog, TF-IDF + linear SVM", 0.8715),
    ("Blog, fine-tuned BERT", 0.9167),
]

## 3. How I measure

The embedder is BAAI/bge-small-en-v1.5 with 33 million parameters, running on CPU through ONNX. It sees 512 tokens, so I take the first three chunks of a résumé, 1,600 characters each, and average their vectors. That gives one vector per résumé.

Two classifiers go on top. The first is k-NN by cosine similarity, where nothing is trained at all. The second is logistic regression over the vectors. For comparison I use TF-IDF with the same regression. Everything runs on five-fold stratified cross-validation.

Every number is computed twice, once on the text as it is and once without the first line and with the category words erased.

In [ ]:
CHUNK, MAX_CHUNKS, BATCH = 1600, 3, 32

def normalise(text):
    return re.sub(r"\s+", " ", text).strip()

def without_title(text, cat):
    parts = re.split(r"\s{3,}", text.strip(), maxsplit=1)
    body = parts[1] if len(parts) > 1 else parts[0]
    for w in category_words(cat):
        pattern = rf"\b{re.escape(w)}\b" if len(w) <= 3 else rf"\b{re.escape(w)}\w*"
        body = re.sub(pattern, " ", body, flags=re.I)
    return normalise(body)

def chunks(text):
    return [text[i:i + CHUNK] for i in range(0, len(text), CHUNK)][:MAX_CHUNKS] or [""]

texts = {
    "with title": [normalise(t) for t in raw],
    "without title": [without_title(t, c) for t, c in zip(raw, y)],
}

In [ ]:
from fastembed import TextEmbedding

model = TextEmbedding("BAAI/bge-small-en-v1.5")

def embed(strings):
    out = []
    for i in range(0, len(strings), BATCH):   # small batches, long chunks are memory hungry
        out.extend(v.tolist() for v in model.embed(strings[i:i + BATCH]))
    return np.array(out, dtype=np.float32)

def embed_documents(docs):
    spans, flat = [], []
    for d in docs:
        c = chunks(d); spans.append((len(flat), len(flat) + len(c))); flat.extend(c)
    vecs = embed(flat)
    x = np.stack([vecs[a:b].mean(axis=0) for a, b in spans])
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-9)

X = {}
for name, docs in texts.items():
    t0 = time.time()
    X[name] = embed_documents(docs)
    print(f"{name}: {X[name].shape} in {(time.time() - t0) / 60:.1f} min")

In [ ]:
ya = np.array(y)

def knn_predict(train_x, train_y, test_x, k=5):
    sims = test_x @ train_x.T
    top = np.argsort(-sims, axis=1)[:, :k]
    preds = []
    for row, idx in zip(sims, top):
        votes = {}
        for j in idx:
            votes[train_y[j]] = votes.get(train_y[j], 0.0) + float(row[j])
        preds.append(max(votes, key=votes.get))
    return np.array(preds)

def tfidf():
    return TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2), max_features=60000)

def cross_validate(x, docs):
    scores = {"bge-small + k-NN": [], "bge-small + logistic regression": [], "TF-IDF + logistic regression": [], "both, averaged probabilities": []}
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(x, ya):
        emb = LogisticRegression(max_iter=2000, C=10).fit(x[tr], ya[tr])
        vec = tfidf(); lex = LogisticRegression(max_iter=2000, C=10).fit(vec.fit_transform([docs[i] for i in tr]), ya[tr])
        lex_x = vec.transform([docs[i] for i in te])
        both = emb.classes_[((emb.predict_proba(x[te]) + lex.predict_proba(lex_x)) / 2).argmax(axis=1)]
        for name, pred in (("bge-small + k-NN", knn_predict(x[tr], ya[tr], x[te])), ("bge-small + logistic regression", emb.predict(x[te])),
                           ("TF-IDF + logistic regression", lex.predict(lex_x)), ("both, averaged probabilities", both)):
            scores[name].append((accuracy_score(ya[te], pred), f1_score(ya[te], pred, average="macro")))
    return {k: (float(np.mean([a for a, _ in v])), float(np.mean([f for _, f in v]))) for k, v in scores.items()}

cv = {name: cross_validate(X[name], texts[name]) for name in texts}
table([{"method": m, "accuracy, with title": f"{cv['with title'][m][0]:.1%}", "macro F1": f"{cv['with title'][m][1]:.3f}",
        "accuracy, without title": f"{cv['without title'][m][0]:.1%}", "macro F1 ": f"{cv['without title'][m][1]:.3f}"} for m in cv["with title"]])

In [ ]:
BLUE, SOFT, GREY, INK, MUTED = "#2a78d6", "#a9c8ee", "#b9bec6", "#0b0b0b", "#52514e"

def hbars(ax, labels, values, colors, title):
    ax.barh(range(len(labels)), values, color=colors, height=0.62)
    ax.set_yticks(range(len(labels)), labels, fontsize=9)
    ax.set_xlim(0, 1.08); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="x", color="#00000012"); ax.set_axisbelow(True); ax.tick_params(length=0, colors=MUTED)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for i, v in enumerate(values): ax.text(v + 0.012, i, f"{v:.0%}", va="center", fontsize=9, color=INK)
    ax.set_title(title, loc="left", fontsize=12, color=INK)

mine = cv["with title"]
bars = [(n, a, GREY) for n, a in others] + [("This notebook, bge-small + k-NN, nothing trained", mine["bge-small + k-NN"][0], SOFT),
                                             ("This notebook, bge-small + logistic regression", mine["bge-small + logistic regression"][0], BLUE)]
bars.sort(key=lambda b: b[1])
fig, ax = plt.subplots(figsize=(8.6, 4.2))
hbars(ax, [b[0] for b in bars], [b[1] for b in bars], [b[2] for b in bars], "Category accuracy, 24 classes, title kept")
plt.tight_layout(); plt.show()

## 4. With the title and without it

I do not think the second variant is the more correct one. In real life the title is in the résumé and there is no reason to throw it away. I need it to see what the result actually rests on.

In [ ]:
names = ["bge-small + k-NN", "bge-small + logistic regression", "TF-IDF + logistic regression"]
fig, ax = plt.subplots(figsize=(8.6, 3.2)); h = 0.36
for j, (variant, color) in enumerate([("with title", BLUE), ("without title", SOFT)]):
    vals = [cv[variant][n][0] for n in names]; ys = [i + (0.5 - j) * h for i in range(len(names))]
    ax.barh(ys, vals, height=h, color=color, label=variant)
    for yy, v in zip(ys, vals): ax.text(v + 0.012, yy, f"{v:.0%}", va="center", fontsize=9, color=INK)
ax.set_yticks(range(len(names)), names, fontsize=9); ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.grid(axis="x", color="#00000012"); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=9)
ax.set_title("What the job title is worth", loc="left", fontsize=12); plt.tight_layout(); plt.show()

With the title, embeddings plus regression give 71% and TF-IDF gives 66%. Without it they swap places, 54% against 60%. Embeddings lose 17 points on the title and TF-IDF loses only 6. It looks like the title is the main carrier of meaning for the embedder. Without it the vector describes office work in general, while TF-IDF still sees the rare professional terms in the body of the résumé.

## 5. Search instead of classification

Nobody predicts categories in my project, people search for people there. So the second check is closer to what I care about. A recruiter types a role, say aviation professional résumé, and gets the ten nearest résumés. I count how many of them belong to the right category.

In [ ]:
def unit(v):
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def precision_at_10(scores):
    top = np.argsort(-scores, axis=1)[:, :10]
    return np.array([(ya[top[i]] == c).mean() for i, c in enumerate(cats)])

role_queries = [f"{c.replace('-', ' ').lower()} professional résumé" for c in cats]
semantic = {name: unit(embed(role_queries)) @ X[name].T for name in texts}
per_cat = precision_at_10(semantic["with title"])
print(f"mean precision@10, with title {per_cat.mean():.1%} | without title {precision_at_10(semantic['without title']).mean():.1%}")

order = np.argsort(per_cat)
fig, ax = plt.subplots(figsize=(8.6, 6))
hbars(ax, [cats[i].replace("-", " ").title() for i in order], per_cat[order],
      [BLUE if v >= 0.7 else SOFT if v >= 0.4 else GREY for v in per_cat[order]], "A recruiter types a role. How many of the top 10 are right")
plt.tight_layout(); plt.show()

The average is 73%. Aviation, chefs, construction, designers, HR and PR are found completely. Consultant, Business Development and Sales are found badly. Résumés from finance, IT and real estate sit under those labels, and by content they are hard to tell from their neighbours. The labelling is the limit here, the search is not.

## 6. A hypothesis about meaning and words

In the project semantics already works together with exact fields, so I tried the same idea on text. I added TF-IDF over the role words to the cosine of the embeddings and merged the two rankings with reciprocal rank fusion. For classification I averaged the probabilities of the two models, that row is already in the table above. I also tried the query prefix that bge was trained with.

In [ ]:
PREFIX = "Represent this sentence for searching relevant passages: "

def rrf(*score_matrices, k=60):
    fused = np.zeros_like(score_matrices[0], dtype=np.float64)
    for s in score_matrices:
        fused += 1.0 / (k + np.argsort(np.argsort(-s, axis=1), axis=1) + 1)
    return fused

search = {}
for name, docs in texts.items():
    vec = tfidf(); doc_matrix = vec.fit_transform(docs)
    words = (vec.transform([c.replace("-", " ").lower() for c in cats]) @ doc_matrix.T).toarray()
    prefixed = unit(embed([PREFIX + q for q in role_queries])) @ X[name].T
    search[name] = {"meaning": precision_at_10(semantic[name]).mean(), "meaning + bge prefix": precision_at_10(prefixed).mean(),
                    "words": precision_at_10(words).mean(), "meaning + words": precision_at_10(rrf(prefixed, words)).mean()}
table([{"role query, precision@10": k, "with title": f"{search['with title'][k]:.1%}", "without title": f"{search['without title'][k]:.1%}"} for k in search["with title"]])

fig, (a, b) = plt.subplots(1, 2, figsize=(9.4, 2.9))
s = search["with title"]
hbars(a, list(s), list(s.values()), [SOFT, SOFT, GREY, BLUE], "Role query, precision@10")
m = cv["with title"]
hbars(b, ["meaning", "words", "meaning + words"], [m["bge-small + logistic regression"][0], m["TF-IDF + logistic regression"][0], m["both, averaged probabilities"][0]],
      [SOFT, GREY, BLUE], "Category accuracy")
plt.tight_layout(); plt.show()

In search it worked, 73% went up to 90%. Almost all of the gain came from the words, the prefix added one point. In classification nothing changed, it stayed at 71%.

Words let you down easily, though. In the variant without the title, search by words gave 0% while semantics gave 42%. The words I needed were erased from the right résumés and stayed in the wrong ones, so the search confidently found the wrong people. In real life this happens when a recruiter types ML engineer and the résumé says data scientist.

## What I take from this

Embeddings without fine-tuning give 71% on this dataset. That is above the popular bag-of-words notebooks and well below fine-tuned models.

Much of any result here rests on the job title. If you compare models on this data, compute both variants.

In search, words together with meaning work better than either alone. Leaving only words is risky, because when the wording differs they return a confident mistake.

The next step for the project is clear. It needs a lexical ranker next to the embeddings, one that yields to semantics when the words are not in the texts.

## Limits

One embedder, one chunking scheme, only the first 4,800 characters of a résumé. The search queries are built from the category names, which is as simple as it gets. The labels are noisy, so 100% is out of reach on them in principle.

The data is real. This notebook shows no fragment of anyone's résumé, only aggregate numbers.

The project lives [here](https://ananasdda.github.io/cv-screener/) and the code is [here](https://github.com/ananasDDA/cv-screener).